In [1]:
import os, logging
os.environ['CUDA_VISIBLE_DEVICES']  = '-1'
os.environ['TF_CPP_MIN_LOG_LEVEL']  = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import warnings
warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.ERROR)
for name in ['lightgbm', 'catboost', 'sklearn']:
    logging.getLogger(name).setLevel(logging.ERROR)

import time
import numpy as np
import pandas as pd
import glob
from pathlib import Path
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import KFold
from scipy.signal import savgol_filter
from numba import njit

INPUT_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction"

# ════════════════════════════════════════════════════════
# V34: V33 + Global pretrained LGBM (all 773 train wells)
#  Speed fix: global dataset build uses skip_pf=True
#  (beam search as PF proxy) -> ~8x faster build.
#  Local model still uses 8-seed PF.
#  Blend: λ=0.55 local + (1-λ)=0.45 global → U-space proj.
# ════════════════════════════════════════════════════════

N_NEIGHBORS   = 15        # spatial neighbor wells (V20 best)
N_TYPEWELLS   = 3         # max typewells in the DTW ensemble (own + neighbors')
K_FOLDS       = 5         # K-fold ensemble (V20 best)

LGBM_N_EST    = 400
CAT_ITERS     = 250
LGBM_DEPTH    = 7
CAT_DEPTH     = 7

# Blend weight search grid for (LGBM weight). CatBoost weight = 1 - w.
BLEND_GRID = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

# Multi-seed PF: run N_PF_SEEDS particle filters with different
# random seeds and average -> more stable TVT estimate
# (top team used 128 seeds; we use 8 as a practical balance)
N_PF_SEEDS  = 8
N_PARTICLES = 128


@njit(cache=True, fastmath=True)
def dtw_predict_numba(query_gr, tw_tvt, tw_gr, radius=50):
    """Returns (predicted_tvt_per_point, mean_alignment_cost)."""
    N = len(query_gr)
    M = len(tw_gr)
    if M == 0 or N == 0:
        out = np.empty(N, np.float32)
        val = tw_tvt[M // 2] if M > 0 else 0.0
        for k in range(N):
            out[k] = val
        return out, 1e18

    qmean = np.mean(query_gr)
    qstd  = np.std(query_gr)
    rmean = np.mean(tw_gr)
    rstd  = np.std(tw_gr)
    qn = (query_gr - qmean) / (qstd + 1e-6)
    rn = (tw_gr - rmean) / (rstd + 1e-6)

    INF = 1e18
    D = np.full((N, M), INF)
    slope = (M - 1.0) / max(N - 1.0, 1.0)

    for i in range(N):
        jc = int(round(i * slope))
        lo = max(0, jc - radius)
        hi = min(M - 1, jc + radius)
        for j in range(lo, hi + 1):
            cost = (qn[i] - rn[j]) ** 2
            if i == 0 and j == 0:
                D[i, j] = cost
            elif i == 0:
                D[i, j] = cost + D[i, j - 1]
            elif j == 0:
                D[i, j] = cost + D[i - 1, j]
            else:
                a = D[i - 1, j - 1]
                b = D[i - 1, j]
                c = D[i, j - 1]
                m = a
                if b < m: m = b
                if c < m: m = c
                D[i, j] = cost + m

    i, j = N - 1, M - 1
    j_map = np.zeros(N, np.int64)
    while i > 0 or j > 0:
        j_map[i] = j
        if i == 0:
            j -= 1
        elif j == 0:
            i -= 1
        else:
            a = D[i - 1, j - 1]
            b = D[i - 1, j]
            c = D[i, j - 1]
            if a <= b and a <= c:
                i -= 1; j -= 1
            elif b <= c:
                i -= 1
            else:
                j -= 1
    j_map[0] = j

    out = np.empty(N, np.float32)
    for k in range(N):
        out[k] = tw_tvt[j_map[k]]

    final_cost = D[N - 1, M - 1] / N
    return out, final_cost


# Warm up numba compilation once (small dummy inputs)
_ = dtw_predict_numba(
    np.zeros(5, np.float32), np.zeros(5, np.float32), np.zeros(5, np.float32)
)


def ncc_predict(query_gr, tw_tvt, tw_gr, window=50):
    """
    Normalized Cross-Correlation TVT estimator.

    For each query point i, find the shift s in tw_gr that
    maximises the NCC between query_gr[i-w:i+w] and
    tw_gr[j-w:j+w], where j tracks the expected position
    (scaled by typewell/query length ratio). The TVT at
    position j+s is returned as the NCC estimate.

    This is complementary to DTW:
      - DTW  : flexible many-to-one warping along full signal
      - NCC  : rigid local shift within a window at each point

    Returns ncc_tvt (n,) float32
    """
    N = len(query_gr)
    M = len(tw_gr)
    if M == 0 or N == 0:
        return np.full(N, tw_tvt[M // 2] if M > 0 else 0.0, np.float32)

    # Normalise both signals
    qn = (query_gr - query_gr.mean()) / (query_gr.std() + 1e-6)
    rn = (tw_gr    - tw_gr.mean())    / (tw_gr.std()    + 1e-6)

    slope  = (M - 1.0) / max(N - 1.0, 1.0)
    result = np.empty(N, np.float32)

    for i in range(N):
        jc = int(round(i * slope))          # expected position in typewell

        # Search window around jc
        lo = max(0, jc - window)
        hi = min(M - 1, jc + window)

        # Local query window
        qi_lo = max(0, i - window // 2)
        qi_hi = min(N - 1, i + window // 2)
        q_seg = qn[qi_lo: qi_hi + 1]
        q_len = len(q_seg)

        if q_len < 3:
            result[i] = tw_tvt[jc]
            continue

        # Slide q_seg over typewell window, compute NCC
        best_corr = -np.inf
        best_j    = jc
        for j in range(lo, hi + 1):
            t_lo = j
            t_hi = min(M - 1, j + q_len - 1)
            t_seg = rn[t_lo: t_hi + 1]
            if len(t_seg) < q_len:
                continue
            corr = float(np.dot(q_seg, t_seg[:q_len]))
            if corr > best_corr:
                best_corr = corr
                best_j    = j + q_len // 2   # centre of match

        best_j = max(0, min(M - 1, best_j))
        result[i] = tw_tvt[best_j]

    return result


@njit(cache=True, fastmath=True)
def beam_search_tvt(query_gr, tw_tvt_sorted, tw_gr_sorted, last_tvt, slope,
                     dmd, n_beams=8, step_candidates=5, max_step_dev=3.0,
                     sigma_obs=15.0):
    """
    Numba-JIT beam search TVT path finder.

    At each MD step, maintain n_beams candidate TVT "states"
    with cumulative cost. Expand each into step_candidates
    next-TVT options around (current_tvt + slope*dmd[i]),
    score each by GR mismatch vs typewell GR(TVT) (linear
    interp), keep the n_beams lowest-cumulative-cost states.

    tw_tvt_sorted, tw_gr_sorted: typewell arrays SORTED by TVT
    (so np.interp works for the GR lookup).

    Returns: bs_tvt (n,) — best path's TVT per MD step.
    """
    n = len(query_gr)
    m = len(tw_tvt_sorted)
    if m == 0:
        out = np.empty(n, np.float32)
        for k in range(n):
            out[k] = last_tvt
        return out

    tvt_lo = tw_tvt_sorted[0]
    tvt_hi = tw_tvt_sorted[-1]

    # beams: (n_beams,) current TVT states
    # paths:  (n_beams, n) TVT history per beam
    beams = np.empty(n_beams)
    for b in range(n_beams):
        beams[b] = last_tvt
    costs = np.zeros(n_beams)
    paths = np.empty((n_beams, n))

    # candidate offsets around the slope-predicted step
    half = step_candidates // 2
    offsets = np.empty(step_candidates)
    for k in range(step_candidates):
        offsets[k] = (k - half) * (max_step_dev / max(half, 1))

    for i in range(n):
        step = slope * dmd[i]
        obs_gr = query_gr[i]

        # candidate next states for each beam: (n_beams * step_candidates,)
        n_cand = n_beams * step_candidates
        cand_tvt  = np.empty(n_cand)
        cand_cost = np.empty(n_cand)
        cand_src  = np.empty(n_cand, np.int64)

        idx = 0
        for b in range(n_beams):
            base = beams[b] + step
            for k in range(step_candidates):
                t = base + offsets[k]
                if t < tvt_lo: t = tvt_lo
                if t > tvt_hi: t = tvt_hi
                exp_gr = np.interp(t, tw_tvt_sorted, tw_gr_sorted)
                step_cost = ((obs_gr - exp_gr) / sigma_obs) ** 2
                cand_tvt[idx]  = t
                cand_cost[idx] = costs[b] + step_cost
                cand_src[idx]  = b
                idx += 1

        # select n_beams lowest cumulative-cost candidates
        order = np.argsort(cand_cost)
        new_beams = np.empty(n_beams)
        new_costs = np.empty(n_beams)
        new_paths = np.empty((n_beams, n))
        for b in range(n_beams):
            sel = order[b]
            src = cand_src[sel]
            new_beams[b] = cand_tvt[sel]
            new_costs[b] = cand_cost[sel]
            # copy path history from source beam, then append new state
            for t in range(i):
                new_paths[b, t] = paths[src, t]
            new_paths[b, i] = cand_tvt[sel]

        beams = new_beams
        costs = new_costs
        paths = new_paths

    best_b = np.argmin(costs)
    out = np.empty(n, np.float32)
    for k in range(n):
        out[k] = paths[best_b, k]
    return out


# Warm up beam search numba compilation
_ = beam_search_tvt(
    np.zeros(5, np.float32),
    np.array([0.0, 1.0, 2.0, 3.0, 4.0]),
    np.array([0.0, 1.0, 2.0, 3.0, 4.0]),
    0.0, 0.0, np.zeros(5, np.float32)
)


@njit(cache=True, fastmath=True)
def particle_filter_tvt_numba(query_gr, tw_tvt_sorted, tw_gr_sorted,
                               last_tvt, slope, dmd,
                               n_particles=128, sigma_process=3.0,
                               sigma_obs=15.0, seed=42):
    """
    Numba-JIT particle filter, sigma values matched to the
    beam search settings (sigma_obs=15.0, step deviation scale
    ~3.0 -> sigma_process=3.0) for an apples-to-apples retry
    after the earlier under-tuned PF attempt (sigma_process=0.5)
    underperformed.

    tw_tvt_sorted, tw_gr_sorted: typewell arrays SORTED by TVT.
    Returns pf_tvt (n,) — weighted-mean TVT estimate per step.
    """
    np.random.seed(seed)
    n = len(query_gr)
    m = len(tw_tvt_sorted)
    if m == 0:
        out = np.empty(n, np.float32)
        for k in range(n):
            out[k] = last_tvt
        return out

    tvt_lo = tw_tvt_sorted[0]
    tvt_hi = tw_tvt_sorted[-1]

    particles = np.empty(n_particles)
    for p in range(n_particles):
        particles[p] = last_tvt + np.random.normal(0.0, sigma_process * 2)
    weights = np.empty(n_particles)
    for p in range(n_particles):
        weights[p] = 1.0 / n_particles

    pf_tvt = np.empty(n, np.float32)

    for i in range(n):
        step = slope * dmd[i]
        obs_gr = query_gr[i]

        # propagate + clip
        for p in range(n_particles):
            particles[p] = particles[p] + step + np.random.normal(0.0, sigma_process)
            if particles[p] < tvt_lo: particles[p] = tvt_lo
            if particles[p] > tvt_hi: particles[p] = tvt_hi

        # reweight by GR likelihood (typewell GR lookup via interp)
        max_log_lik = -1e18
        log_liks = np.empty(n_particles)
        for p in range(n_particles):
            exp_gr = np.interp(particles[p], tw_tvt_sorted, tw_gr_sorted)
            ll = -0.5 * ((obs_gr - exp_gr) / sigma_obs) ** 2
            log_liks[p] = ll
            if ll > max_log_lik:
                max_log_lik = ll

        w_sum = 0.0
        for p in range(n_particles):
            w = np.exp(log_liks[p] - max_log_lik) * weights[p]
            weights[p] = w
            w_sum += w

        if w_sum < 1e-12:
            for p in range(n_particles):
                weights[p] = 1.0 / n_particles
            w_sum = 1.0
        else:
            for p in range(n_particles):
                weights[p] = weights[p] / w_sum

        # weighted mean estimate
        est = 0.0
        for p in range(n_particles):
            est += particles[p] * weights[p]
        pf_tvt[i] = est

        # effective sample size -> resample if needed
        sq_sum = 0.0
        for p in range(n_particles):
            sq_sum += weights[p] ** 2
        n_eff = 1.0 / sq_sum if sq_sum > 0 else n_particles

        if n_eff < n_particles / 2:
            cdf = np.empty(n_particles)
            running = 0.0
            for p in range(n_particles):
                running += weights[p]
                cdf[p] = running
            new_particles = np.empty(n_particles)
            start = np.random.uniform(0.0, 1.0 / n_particles)
            j = 0
            for p in range(n_particles):
                u = start + p / n_particles
                while j < n_particles - 1 and u > cdf[j]:
                    j += 1
                new_particles[p] = particles[j]
            particles = new_particles
            for p in range(n_particles):
                weights[p] = 1.0 / n_particles

    return pf_tvt


# Warm up PF numba compilation
_ = particle_filter_tvt_numba(
    np.zeros(5, np.float32),
    np.array([0.0, 1.0, 2.0, 3.0, 4.0]),
    np.array([0.0, 1.0, 2.0, 3.0, 4.0]),
    0.0, 0.0, np.zeros(5, np.float32)
)


# ── Spatial Index ─────────────────────────────────────
train_files = sorted(glob.glob(f"{INPUT_DIR}/train/*__horizontal_well.csv"))
test_files  = sorted(glob.glob(f"{INPUT_DIR}/test/*__horizontal_well.csv"))


def find_typewell_path(directory, well_id):
    """Typewell files are named <well_id>__typewell__<typewell_id>.csv
    or sometimes <well_id>__typewell.csv. Find whichever exists."""
    matches = glob.glob(f"{directory}/{well_id}__typewell*.csv")
    return matches[0] if matches else None


train_centers = []
for f in train_files:
    df = pd.read_csv(f, usecols=['X', 'Y'])
    wid = Path(f).stem.split("__")[0]
    train_centers.append({'well_id': wid,
                           'X': df['X'].mean(), 'Y': df['Y'].mean()})
train_centers_df = pd.DataFrame(train_centers)
nbrs = NearestNeighbors(n_neighbors=N_NEIGHBORS, algorithm='ball_tree').fit(
    train_centers_df[['X', 'Y']].values)

# Also build a neighbor index restricted to N_TYPEWELLS (for typewell
# borrowing), separate from the N_NEIGHBORS used for local training rows.
nbrs_tw = NearestNeighbors(
    n_neighbors=min(N_TYPEWELLS, len(train_centers_df)), algorithm='ball_tree'
).fit(train_centers_df[['X', 'Y']].values)


# ── Geology label -> integer code map ──────────────────
# Build once from a sample of typewell files (train + test) so
# the mapping is consistent across all wells. Geology labels
# are a small fixed vocabulary (formation names), so sampling
# a subset of typewells is sufficient to discover them all.
def _build_geology_code_map():
    labels = set()
    sample_paths = (
        glob.glob(f"{INPUT_DIR}/train/*__typewell*.csv")[:200]
        + glob.glob(f"{INPUT_DIR}/test/*__typewell*.csv")
    )
    for p in sample_paths:
        try:
            col = pd.read_csv(p, usecols=['Geology'])['Geology']
            labels.update(str(v) for v in col.dropna().unique())
        except Exception:
            continue
    return {label: i for i, label in enumerate(sorted(labels))}


GEOLOGY_CODE_MAP = _build_geology_code_map()
print(f"Geology labels found: {len(GEOLOGY_CODE_MAP)} -> {GEOLOGY_CODE_MAP}")


# Cache typewell dataframes by file path (avoid re-reading/sorting)
_tw_cache = {}


def load_typewell(directory, well_id):
    path = find_typewell_path(directory, well_id)
    if path is None:
        return None
    if path not in _tw_cache:
        try:
            _tw_cache[path] = pd.read_csv(path).sort_values('TVT').reset_index(drop=True)
        except Exception:
            _tw_cache[path] = None
    return _tw_cache[path]


def get_typewell_candidates(directory, well_id, center_xy, max_n=N_TYPEWELLS):
    """Own typewell + nearest spatial-neighbor wells' typewells,
    deduplicated by file path, up to max_n total."""
    candidates = []
    seen_paths = set()

    own_path = find_typewell_path(directory, well_id)
    if own_path is not None:
        tw = load_typewell(directory, well_id)
        if tw is not None:
            candidates.append(tw)
            seen_paths.add(own_path)

    if len(candidates) < max_n:
        _, idx = nbrs_tw.kneighbors(np.array(center_xy).reshape(1, -1))
        for nidx in idx[0]:
            if len(candidates) >= max_n:
                break
            nw_id = train_centers_df.iloc[nidx]['well_id']
            npath = find_typewell_path(f"{INPUT_DIR}/train", nw_id)
            if npath is None or npath in seen_paths:
                continue
            tw = load_typewell(f"{INPUT_DIR}/train", nw_id)
            if tw is not None:
                candidates.append(tw)
                seen_paths.add(npath)

    return candidates


# ════════════════════════════════════════════════════════
# Features — V18 set + multi-typewell DTW ensemble features
#   dtw_d_best:  dtw_d from the typewell with lowest alignment
#                cost (best GR-shape match)
#   dtw_d_mean:  mean dtw_d across all typewell candidates
#                (ensemble / robust estimate)
#   dtw_cost_best: alignment cost of the best-matching typewell
#                (lower = more confident match; useful as a
#                 "confidence" signal for the model)
# ════════════════════════════════════════════════════════
FEATURES = [
    'md_since', 'frac', 'frac2', 'sqrt_frac',
    'z', 'dz',
    'gr', 'gr_d1', 'gr_rolling5', 'gr_rolling21',
    'gr_std5', 'gr_std21',
    'gr_rolling_max5', 'gr_rolling_min5',
    'gr_rolling_max21', 'gr_rolling_min21',
    'dtw_d_best', 'dtw_d_mean', 'dtw_cost_best',
    'bs_d',        # beam search TVT delta
    'pf_d',        # particle filter TVT delta
    'geology_code', # NEW: formation label at DTW-predicted TVT
    'slp_d',
    'tvt_ff_d',
    'tw_tvt_mean', 'tw_tvt_std',
    'tw_gr_mean',  'tw_gr_std',
    'gr_dev',      'gr_zscore',
    'dist_anchor', 'dist_anchor_norm', 'known_len', 'last_tvt',
]


def build_features(hw, tw_candidates, tvt_col='TVT', last_tvt_override=None,
                   skip_pf=False):
    """tw_candidates: list of typewell DataFrames (1 to N_TYPEWELLS).
    skip_pf=True: skip particle filter (for fast global dataset build)."""
    n   = len(hw)
    tvt = hw[tvt_col].values.astype(float) \
          if tvt_col in hw.columns else np.full(n, np.nan)
    md  = hw['MD'].values.astype(float)
    z   = hw['Z'].values.astype(float)
    gr  = hw['GR'].interpolate(
        limit_direction='both').fillna(0).values.astype(float)

    tvt_in    = hw['TVT_input'].values.astype(float) \
                if 'TVT_input' in hw.columns else tvt.copy()
    known_idx = np.where(~np.isnan(tvt_in))[0]
    ps_idx    = known_idx[-1] if len(known_idx) > 0 else 0
    last_tvt  = last_tvt_override if last_tvt_override is not None \
                else (tvt_in[ps_idx] if len(known_idx) > 0 else float(np.nanmean(tvt)))

    # Slope from known portion
    if len(known_idx) >= 5:
        ti    = known_idx[-20:]
        slope = np.polyfit(md[ti], tvt_in[ti], 1)[0] \
                if np.std(md[ti]) > 1e-6 else 0.0
    else:
        slope = 0.0

    # ── Multi-typewell DTW ensemble ──────────────────────
    gr32 = gr.astype(np.float32)
    if tw_candidates:
        preds = []
        costs = []
        # Use the FIRST candidate (own typewell) for the global
        # typewell-stat features (tw_tvt_mean etc.) to keep those
        # features stable / well-defined.
        primary_tw = tw_candidates[0]
        tw_tvt_v = primary_tw['TVT'].values.astype(np.float32)
        tw_gr_v  = primary_tw['GR'].values.astype(np.float32)
        tw_gr_m  = float(np.nanmean(tw_gr_v))
        tw_gr_s  = float(np.nanstd(tw_gr_v))
        tw_tv_m  = float(tw_tvt_v.mean())
        tw_tv_s  = float(tw_tvt_v.std())

        for cand in tw_candidates:
            c_tvt = cand['TVT'].values.astype(np.float32)
            c_gr  = cand['GR'].values.astype(np.float32)
            pred, cost = dtw_predict_numba(gr32, c_tvt, c_gr)
            preds.append(pred)
            costs.append(cost)

        preds = np.stack(preds, axis=0)       # (n_candidates, n)
        costs = np.array(costs)               # (n_candidates,)
        best_i = int(np.argmin(costs))
        dtw_pred_best = preds[best_i]
        dtw_pred_mean = preds.mean(axis=0)
        dtw_cost_best = float(costs[best_i])

        # ── Beam Search (use best-matching typewell) ──────
        best_tw = tw_candidates[best_i]
        bs_tvt_sorted = best_tw['TVT'].values.astype(np.float32)
        bs_gr_sorted  = best_tw['GR'].values.astype(np.float32)
        # tw_candidates are already sorted by TVT (loaded that way)
        dmd = np.diff(md, prepend=md[0]).astype(np.float32)
        bs_result = beam_search_tvt(
            gr32, bs_tvt_sorted, bs_gr_sorted,
            np.float32(last_tvt), np.float32(slope), dmd,
            n_beams=8, step_candidates=5, max_step_dev=3.0, sigma_obs=15.0
        )

        # ── Particle Filter: multi-seed ensemble ──────────
        # Run N_PF_SEEDS PFs with different random seeds and
        # average -> reduces seed-dependent noise (top team
        # used 128 seeds; we use N_PF_SEEDS=8 for speed).
        if skip_pf:
            pf_result = bs_result.copy()  # use beam search as PF proxy
        else:
            pf_runs = []
            for _seed in range(N_PF_SEEDS):
                pf_run = particle_filter_tvt_numba(
                    gr32, bs_tvt_sorted, bs_gr_sorted,
                    np.float32(last_tvt), np.float32(slope), dmd,
                    n_particles=N_PARTICLES, sigma_process=3.0,
                    sigma_obs=15.0, seed=_seed * 7 + 1
                )
                pf_runs.append(pf_run)
            pf_result = np.mean(pf_runs, axis=0).astype(np.float32)

        # ── Geology formation label at DTW-predicted TVT ──
        # Look up the typewell's Geology column at the TVT
        # position predicted by DTW (best-matching typewell).
        # Encoded as an integer category code (consistent
        # mapping built once globally, see GEOLOGY_CODE_MAP).
        if 'Geology' in best_tw.columns:
            geo_vals = best_tw['Geology'].values
            geo_tvt  = best_tw['TVT'].values.astype(float)
            # nearest-TVT lookup for each predicted dtw TVT value
            sort_idx = np.argsort(geo_tvt)
            geo_tvt_sorted = geo_tvt[sort_idx]
            geo_vals_sorted = geo_vals[sort_idx]
            insert_pos = np.searchsorted(geo_tvt_sorted, dtw_pred_best)
            insert_pos = np.clip(insert_pos, 0, len(geo_tvt_sorted) - 1)
            geology_at_pred = geo_vals_sorted[insert_pos]
            geology_code = np.array(
                [GEOLOGY_CODE_MAP.get(str(g), -1) for g in geology_at_pred],
                dtype=np.float32)
        else:
            geology_code = np.full(n, -1.0, dtype=np.float32)

    else:
        dtw_pred_best = np.full(n, last_tvt, np.float32)
        dtw_pred_mean = dtw_pred_best.copy()
        dtw_cost_best = 1e18
        bs_result = np.full(n, last_tvt, np.float32)
        pf_result = np.full(n, last_tvt, np.float32)
        geology_code = np.full(n, -1.0, dtype=np.float32)
        tw_gr_m = tw_gr_s = tw_tv_m = tw_tv_s = 0.0

    # GR rolling (mean / std)
    gr_s   = pd.Series(gr)
    gr_r5  = gr_s.rolling(5,  min_periods=1).mean().values
    gr_r21 = gr_s.rolling(21, min_periods=1).mean().values
    gr_s5  = gr_s.rolling(5,  min_periods=1).std().fillna(0).values
    gr_s21 = gr_s.rolling(21, min_periods=1).std().fillna(0).values

    # GR rolling local extremes
    gr_max5  = gr_s.rolling(5,  min_periods=1, center=True).max().values
    gr_min5  = gr_s.rolling(5,  min_periods=1, center=True).min().values
    gr_max21 = gr_s.rolling(21, min_periods=1, center=True).max().values
    gr_min21 = gr_s.rolling(21, min_periods=1, center=True).min().values

    # Forward fill (stable — doesn't compound)
    tvt_ff = pd.Series(tvt_in).ffill().bfill().values

    # Anchor distance
    if len(known_idx) > 0:
        dist_anc = np.min(np.abs(
            np.arange(n)[:, None] - known_idx[None, :]), axis=1).astype(float)
    else:
        dist_anc = np.full(n, float(n))
    dist_anc_norm = dist_anc / max(n, 1)

    # Fraction
    n_unk = max(n - ps_idx - 1, 1)
    frac  = np.zeros(n)
    frac[ps_idx + 1:] = np.arange(1, n - ps_idx) / n_unk

    df = pd.DataFrame({
        'md_since':    md - md[ps_idx],
        'frac':        frac,
        'frac2':       frac ** 2,
        'sqrt_frac':   np.sqrt(frac),
        'z':           z,
        'dz':          np.diff(z, prepend=z[0]),
        'gr':          gr,
        'gr_d1':       np.diff(gr, prepend=gr[0]),
        'gr_rolling5': gr_r5,
        'gr_rolling21': gr_r21,
        'gr_std5':     gr_s5,
        'gr_std21':    gr_s21,
        'gr_rolling_max5':  gr_max5,
        'gr_rolling_min5':  gr_min5,
        'gr_rolling_max21': gr_max21,
        'gr_rolling_min21': gr_min21,
        'dtw_d_best':  dtw_pred_best - last_tvt,
        'dtw_d_mean':  dtw_pred_mean - last_tvt,
        'dtw_cost_best': dtw_cost_best,
        'bs_d':        bs_result - last_tvt,   # beam search delta
        'pf_d':        pf_result - last_tvt,   # particle filter delta
        'geology_code': geology_code,          # NEW: formation label code
        'slp_d':       last_tvt + slope * (md - md[ps_idx]) - last_tvt,
        'tvt_ff_d':    tvt_ff - last_tvt,   # stable fill delta
        'tw_tvt_mean': tw_tv_m,
        'tw_tvt_std':  tw_tv_s,
        'tw_gr_mean':  tw_gr_m,
        'tw_gr_std':   tw_gr_s,
        'gr_dev':      gr - tw_gr_m,
        'gr_zscore':   (gr - tw_gr_m) / (tw_gr_s + 1e-6),
        'dist_anchor': dist_anc,
        'dist_anchor_norm': dist_anc_norm,
        'known_len':   len(known_idx),
        'last_tvt':    last_tvt,
        'target':      tvt - last_tvt,
    })
    return df, ps_idx, last_tvt, slope


def make_models(seed=42):
    lgbm = LGBMRegressor(n_estimators=LGBM_N_EST, learning_rate=0.05,
                          max_depth=LGBM_DEPTH, num_leaves=127,
                          min_child_samples=10, subsample=0.8,
                          random_state=seed, n_jobs=-1, verbose=-1)
    cat = CatBoostRegressor(iterations=CAT_ITERS, learning_rate=0.05,
                             depth=CAT_DEPTH, random_seed=seed, verbose=0,
                             thread_count=-1)
    return lgbm, cat


def kfold_predict(X_tr, y_tr, X_pred, k=K_FOLDS):
    """
    Fit K folds of (LGBM, CatBoost) on (X_tr, y_tr) and return
    averaged predictions on X_pred for each model type.
    Reduces variance vs a single train/predict split.
    Returns (lgbm_pred_avg, cat_pred_avg).
    """
    n = len(X_tr)
    k_eff = max(2, min(k, n))  # KFold needs >=2 splits and <= n samples
    kf = KFold(n_splits=k_eff, shuffle=True, random_state=42)

    lgbm_preds = []
    cat_preds  = []
    for fold_i, (tr_idx, _) in enumerate(kf.split(X_tr)):
        lgbm, cat = make_models(seed=42 + fold_i)
        lgbm.fit(X_tr[tr_idx], y_tr[tr_idx])
        cat.fit(X_tr[tr_idx], y_tr[tr_idx])
        lgbm_preds.append(lgbm.predict(X_pred))
        cat_preds.append(cat.predict(X_pred))

    return np.mean(lgbm_preds, axis=0), np.mean(cat_preds, axis=0)


def smooth_delta(delta, known_mask):
    """
    Smooth the predicted DELTA (pred - last_tvt) for the
    UNKNOWN portion only, using a Savitzky-Golay filter.
    """
    out = delta.copy()
    unk_idx = np.where(~known_mask)[0]
    if len(unk_idx) >= 5:
        seg = delta[unk_idx]
        win = min(11, len(seg))
        if win % 2 == 0:
            win -= 1
        if win >= 5:
            seg_smooth = savgol_filter(seg, win, 3)
            out[unk_idx] = seg_smooth
    return out


def uspace_projection(tvt_pred, md, z, known_mask, last_tvt,
                       ps_idx, beta=0.75, poly_deg=4):
    """
    U-space polynomial projection post-processing.
    From top-team approach (8.3 score):

    U = TVT + Z  (stratigraphic space - removes vertical depth
                  trend, leaving only geological position signal)
    A_w = last_tvt + z[ps_idx]  (anchor in U-space)
    s   = normalised MD in unknown zone [0, 1]

    Fit a degree-4 robust polynomial to U_centered = U - A_w
    in the unknown region, then blend:
      tvt_proj = A_w + U_poly(s) - z
      tvt_final = (1 - beta) * tvt_pred + beta * tvt_proj

    beta=0.75 (top team's value) means heavy projection weight.
    """
    n = len(tvt_pred)
    unk_idx = np.where(~known_mask)[0]
    if len(unk_idx) < poly_deg + 2:
        return tvt_pred.copy()

    # U-space anchor
    A_w = last_tvt + z[ps_idx]

    # U values from current prediction
    U = tvt_pred + z

    # Normalised MD in unknown zone
    md_start = md[ps_idx]
    md_end   = md[-1]
    md_range = max(md_end - md_start, 1.0)
    s_all = (md - md_start) / md_range

    # Fit polynomial on unknown portion only
    s_unk = s_all[unk_idx]
    U_unk = U[unk_idx] - A_w   # centred

    try:
        coeffs = np.polyfit(s_unk, U_unk, deg=poly_deg)
    except Exception:
        return tvt_pred.copy()

    # Project back to TVT
    U_proj_all = np.polyval(coeffs, s_all) + A_w
    tvt_proj   = U_proj_all - z

    # Blend: only apply in unknown zone (keep known exact)
    tvt_final = tvt_pred.copy()
    tvt_final[unk_idx] = ((1 - beta) * tvt_pred[unk_idx]
                           + beta * tvt_proj[unk_idx])
    return tvt_final


# ════════════════════════════════════════════════════════
# Validate (random 50 train wells, with blend-weight search)
# Random selection -> better distribution coverage than
# first-N (which may be geographically/geologically biased)
# ════════════════════════════════════════════════════════
N_VAL_WELLS = 20
rng_val = np.random.default_rng(seed=123)
val_indices = sorted(rng_val.choice(len(train_files), size=min(N_VAL_WELLS, len(train_files)), replace=False))
val_files   = [train_files[i] for i in val_indices]

t0 = time.time()
print(f"=== Validating on {N_VAL_WELLS} random train wells (multi-typewell + K-fold) ===")

val_data = []  # (wid, lgbm_d, cat_d, known, hw, last_tvt)

for f in val_files:
    wid = Path(f).stem.split("__")[0]
    hw  = pd.read_csv(f).reset_index(drop=True)
    if 'TVT' not in hw.columns: continue

    center_xy = hw[['X', 'Y']].mean().values
    tw_candidates = get_typewell_candidates(f"{INPUT_DIR}/train", wid, center_xy)

    ps_idx = hw['TVT_input'].last_valid_index()
    if ps_idx is None or ps_idx >= len(hw) - 1: continue

    feat, ps_idx, last_tvt, slope = build_features(hw, tw_candidates, 'TVT')

    known = hw['TVT_input'].notna().values
    X_tr  = feat.loc[known, FEATURES].fillna(0).values
    y_tr  = feat.loc[known, 'target'].values
    if len(X_tr) < 20: continue

    X_all = feat[FEATURES].fillna(0).values
    lgbm_d, cat_d = kfold_predict(X_tr, y_tr, X_all, k=K_FOLDS)

    md_vals = hw['MD'].values.astype(float)
    z_vals  = hw['Z'].values.astype(float)
    val_data.append((wid, lgbm_d, cat_d, known, hw, last_tvt, md_vals, z_vals, ps_idx))
    print(f"  {wid}: typewells used = {len(tw_candidates)}")

BETA_GRID    = [0.85, 0.90, 0.95, 1.00]
POLYDEG_GRID = [2, 3]

best_w       = 0.50
best_beta    = 0.75
best_deg     = 4
best_rmse    = None

print("=== Grid search: LGBM weight × beta × poly_deg ===")
for w in BLEND_GRID:
    for beta in BETA_GRID:
        for deg in POLYDEG_GRID:
            rmses = []
            for wid, lgbm_d, cat_d, known, hw, last_tvt, md_vals, z_vals, ps_idx in val_data:
                delta = w * lgbm_d + (1 - w) * cat_d
                delta = smooth_delta(delta, known)
                tvt_pred = last_tvt + delta
                tvt_pred = np.where(known, hw['TVT_input'].values, tvt_pred)
                tvt_pred = uspace_projection(tvt_pred, md_vals, z_vals,
                                             known, last_tvt, ps_idx,
                                             beta=beta, poly_deg=deg)
                tvt_pred = np.where(known, hw['TVT_input'].values, tvt_pred)
                unknown = ~known
                actual  = hw['TVT'].values[unknown]
                pred    = tvt_pred[unknown]
                valid   = ~np.isnan(actual)
                if valid.sum() < 5: continue
                rmses.append(np.sqrt(np.mean((actual[valid] - pred[valid]) ** 2)))
            mean_rmse = np.mean(rmses) if rmses else np.inf
            print(f"  w={w:.2f} beta={beta:.2f} deg={deg} -> RMSE={mean_rmse:.3f}")
            if best_rmse is None or mean_rmse < best_rmse:
                best_rmse = mean_rmse
                best_w    = w
                best_beta = beta
                best_deg  = deg

print(f"\nBest: LGBM_w={best_w:.2f}, beta={best_beta:.2f}, poly_deg={best_deg}")
print(f"Best Mean RMSE: {best_rmse:.3f}")
print(f"Previous: V31 LB=14.698 (best), V32=14.702 (beta=0.95,deg=2)")
print(f"Validation time: {time.time() - t0:.1f}s")

# Per-well RMSE at best params
print("\nPer-well RMSE at best params:")
for wid, lgbm_d, cat_d, known, hw, last_tvt, md_vals, z_vals, ps_idx in val_data:
    delta = best_w * lgbm_d + (1 - best_w) * cat_d
    delta = smooth_delta(delta, known)
    tvt_pred = last_tvt + delta
    tvt_pred = np.where(known, hw['TVT_input'].values, tvt_pred)
    tvt_pred = uspace_projection(tvt_pred, md_vals, z_vals,
                                 known, last_tvt, ps_idx,
                                 beta=best_beta, poly_deg=best_deg)
    tvt_pred = np.where(known, hw['TVT_input'].values, tvt_pred)
    unknown = ~known
    actual  = hw['TVT'].values[unknown]
    pred    = tvt_pred[unknown]
    valid   = ~np.isnan(actual)
    if valid.sum() < 5: continue
    rmse = np.sqrt(np.mean((actual[valid] - pred[valid]) ** 2))
    print(f"  {wid}: RMSE = {rmse:.3f}")

# ════════════════════════════════════════════════════════
# GLOBAL PRETRAINED LGBM
# Train one LGBM on ALL 773 train wells (GroupKFold=5).
# This gives a global trajectory estimate that complements
# the local neighbor model — top team uses this as an
# independent "engine" and blends with local model at λ=0.55.
# We use a lighter model (n_estimators=300) to keep runtime
# manageable. The global model predictions are stored in
# global_model_list and averaged at test time.
# ════════════════════════════════════════════════════════
from sklearn.model_selection import GroupKFold

GLOBAL_LAMBDA   = 0.55   # weight for local model; (1-lambda) for global
GLOBAL_N_EST    = 300    # lighter than local model for speed

print("\n=== Building global training dataset (all train wells) ===")
t_global = time.time()

all_global_dfs = []
for i, f in enumerate(train_files):
    wid = Path(f).stem.split("__")[0]
    try:
        hw = pd.read_csv(f).reset_index(drop=True)
        if 'TVT' not in hw.columns: continue
        center_xy = hw[['X', 'Y']].mean().values
        tw_candidates = get_typewell_candidates(f"{INPUT_DIR}/train", wid, center_xy)
        feat, _, _, _ = build_features(hw, tw_candidates, 'TVT',
                                        skip_pf=True)  # PF skip for speed
        feat['well_id'] = wid
        all_global_dfs.append(feat)
    except Exception:
        continue
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(train_files)} wells processed... ({time.time()-t_global:.0f}s)")

global_df = pd.concat(all_global_dfs, ignore_index=True)
global_mask = global_df['target'].notna()
X_global = global_df.loc[global_mask, FEATURES].fillna(0).values
y_global = global_df.loc[global_mask, 'target'].values
g_global = global_df.loc[global_mask, 'well_id'].values
print(f"  Global training rows: {len(X_global):,} from {global_df['well_id'].nunique()} wells")
print(f"  Global dataset build time: {time.time()-t_global:.1f}s")

# Train global LGBM with GroupKFold (wells as groups)
print("\n=== Training global LGBM (GroupKFold=5) ===")
t_gtrain = time.time()
gkf = GroupKFold(n_splits=5)
global_lgbm_models = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_global, y_global, g_global)):
    g_lgbm = LGBMRegressor(
        n_estimators=GLOBAL_N_EST, learning_rate=0.05,
        max_depth=7, num_leaves=127,
        min_child_samples=15, subsample=0.8,
        colsample_bytree=0.8, random_state=42,
        n_jobs=-1, verbose=-1
    )
    g_lgbm.fit(
        X_global[tr_idx], y_global[tr_idx],
        eval_set=[(X_global[va_idx], y_global[va_idx])],
        eval_metric='rmse',
        callbacks=[__import__('lightgbm').early_stopping(30, verbose=False)]
    )
    oof_pred = g_lgbm.predict(X_global[va_idx])
    oof_rmse = np.sqrt(np.mean((y_global[va_idx] - oof_pred) ** 2))
    global_lgbm_models.append(g_lgbm)
    print(f"  Global LGBM Fold {fold+1}: OOF RMSE={oof_rmse:.3f}  ({time.time()-t_gtrain:.0f}s)")

print(f"  Global model training time: {time.time()-t_gtrain:.1f}s")

def predict_global(X_test):
    """Average predictions across all global fold models."""
    preds = np.array([m.predict(X_test) for m in global_lgbm_models])
    return preds.mean(axis=0)

# ════════════════════════════════════════════════════════
# Test
# ════════════════════════════════════════════════════════

sample_sub = pd.read_csv(f"{INPUT_DIR}/sample_submission.csv")
sample_sub['well_id'] = sample_sub['id'].str.rsplit('_', n=1).str[0]
sample_sub['row_idx'] = sample_sub['id'].str.rsplit('_', n=1).str[1].astype(int)
all_preds = []

# Cache: avoid recomputing features for the same neighbor train well
# across multiple test wells.
_feat_cache = {}


def get_train_feat(nw_id):
    if nw_id not in _feat_cache:
        try:
            df = pd.read_csv(f"{INPUT_DIR}/train/{nw_id}__horizontal_well.csv")
        except FileNotFoundError:
            _feat_cache[nw_id] = None
            return None
        if 'TVT' not in df.columns:
            _feat_cache[nw_id] = None
            return None
        center_xy = df[['X', 'Y']].mean().values
        tw_candidates = get_typewell_candidates(f"{INPUT_DIR}/train", nw_id, center_xy)
        feat, _, _, _ = build_features(df, tw_candidates, 'TVT')
        _feat_cache[nw_id] = feat
    return _feat_cache[nw_id]


t1 = time.time()
print("\n=== Test Prediction ===")
for test_file in test_files:
    well_id = Path(test_file).stem.split("__")[0]
    test_df = pd.read_csv(test_file).reset_index(drop=True)

    center_xy = test_df[['X', 'Y']].mean().values
    tw_candidates = get_typewell_candidates(f"{INPUT_DIR}/test", well_id, center_xy)
    # If no own test typewell found, fall back to nearest train typewells only
    if not tw_candidates:
        _, idx = nbrs_tw.kneighbors(np.array(center_xy).reshape(1, -1))
        for nidx in idx[0][:N_TYPEWELLS]:
            nw_id = train_centers_df.iloc[nidx]['well_id']
            tw = load_typewell(f"{INPUT_DIR}/train", nw_id)
            if tw is not None:
                tw_candidates.append(tw)

    ps_idx   = test_df['TVT_input'].last_valid_index()
    last_tvt = float(test_df.loc[ps_idx, 'TVT_input'])

    tc = test_df[['X', 'Y']].mean().values.reshape(1, -1)
    _, idx = nbrs.kneighbors(tc)
    nearest = train_centers_df.iloc[idx[0]]['well_id'].tolist()

    local_dfs = []
    for nw_id in nearest:
        feat = get_train_feat(nw_id)
        if feat is not None:
            local_dfs.append(feat)

    if not local_dfs: continue
    local = pd.concat(local_dfs, ignore_index=True).dropna(subset=['target'])
    X_tr  = local[FEATURES].fillna(0).values
    y_tr  = local['target'].values

    feat_t, ps_idx, last_tvt, _ = build_features(
        test_df, tw_candidates, 'TVT_input', last_tvt)
    X_test = feat_t[FEATURES].fillna(0).values

    # Local model prediction
    lgbm_d, cat_d = kfold_predict(X_tr, y_tr, X_test, k=K_FOLDS)
    local_delta = best_w * lgbm_d + (1 - best_w) * cat_d

    # Global model prediction
    global_delta = predict_global(X_test)

    # Blend: λ * local + (1-λ) * global (top team λ=0.55)
    delta = GLOBAL_LAMBDA * local_delta + (1 - GLOBAL_LAMBDA) * global_delta

    known_mask = ~np.isnan(test_df['TVT_input'].values)
    delta = smooth_delta(delta, known_mask)

    tvt_pred = last_tvt + delta
    tvt_pred = np.where(known_mask, test_df['TVT_input'].values, tvt_pred)

    # U-space polynomial projection
    md_t = test_df['MD'].values.astype(float)
    z_t  = test_df['Z'].values.astype(float)
    tvt_pred = uspace_projection(tvt_pred, md_t, z_t,
                                  known_mask, last_tvt, ps_idx,
                                  beta=best_beta, poly_deg=best_deg)
    tvt_pred = np.where(known_mask, test_df['TVT_input'].values, tvt_pred)

    well_sub = sample_sub[sample_sub['well_id'] == well_id].copy()
    well_sub = well_sub.sort_values('row_idx').reset_index(drop=True)
    well_sub['tvt'] = well_sub['row_idx'].apply(
        lambda i: float(tvt_pred[i]) if i < len(tvt_pred) else np.nan)
    well_sub['tvt'] = well_sub['tvt'].fillna(well_sub['tvt'].median())
    all_preds.append(well_sub[['id', 'tvt']])
    print(f"  {well_id}: Done ✅  TVT=[{tvt_pred.min():.1f},{tvt_pred.max():.1f}]  "
          f"(typewells={len(tw_candidates)})")

submission = pd.concat(all_preds, ignore_index=True)
submission['tvt'] = submission['tvt'].fillna(submission['tvt'].median())
submission.to_csv('submission.csv', index=False)
print("\nDone!", submission.shape)
print("NaN count:", submission['tvt'].isna().sum())
print(f"Test prediction time: {time.time() - t1:.1f}s")
print(f"Total time: {time.time() - t0:.1f}s")
print(submission.head(10))

Geology labels found: 43 -> {'AC_UEF_BHL': 0, 'AC_UEF_THL': 1, 'AC_UEF_TRGT': 2, 'ANCC': 3, 'ASTNL': 4, 'ASTNU': 5, 'BUDA': 6, 'Clay Rich Interval': 7, 'EGFD100': 8, 'EGFD200': 9, 'EGFD300': 10, 'EGFD300b': 11, 'EGFD300c': 12, 'EGFD400': 13, 'EGFDL': 14, 'EGFDU': 15, 'EGFD_IPT': 16, 'LBHL': 17, 'LL TGT': 18, 'LL THL': 19, 'LLEF BHL': 20, 'LLEF TGT': 21, 'LLEF THL': 22, 'LL_BHL': 23, 'LL_TGT': 24, 'LL_THL': 25, 'LTGT': 26, 'LTHL': 27, 'MNSS': 28, 'OLMOS': 29, 'UBHL': 30, 'UEGFD BHL': 31, 'UEGFD TGT': 32, 'UEGFD THL': 33, 'ULEF BHL': 34, 'ULEF TGT': 35, 'ULEF THL': 36, 'UL_BHL': 37, 'UL_TGT': 38, 'UL_THL': 39, 'UPSN': 40, 'UTGT': 41, 'UTHL': 42}
=== Validating on 20 random train wells (multi-typewell + K-fold) ===
  0498acab: typewells used = 3
  1131525f: typewells used = 3
  2d2d0c6b: typewells used = 3
  2e63d9de: typewells used = 3
  38991fd4: typewells used = 3
  42669188: typewells used = 3
  46dfcfca: typewells used = 3
  54a7e3c2: typewells used = 3
  58fa8486: typewells used = 3